# Results

Merges `03_results_process`, `04_results_individual_profile` and
`05_results_complete_energy_profile`. Three evaluations, one notebook.

Each aggregates in **two stages**, so the unit named below is the unit of the
*last* one — the population section 3 counts:

| # | Evaluation | Stage 1 — one value per… | Stage 2 (the tables) — median over… | Direction |
|---|---|---|---|---|
| **P** timing | Process timing errors | **case** — read straight from `process_eval_per_case.parquet`, the raw per-case sample the pipeline's own medians were built from | all test **cases**, pooled (`Span WAPE` is a sum-over-sum across them) | **lower** |
| **P** discovery | Fitness / Precision / Generalization / Simplicity | **process** — model-level: the real log replayed against the net, per station, averaged. No per-case meaning exists | the **processes** | **higher** |
| **I** | Individual profile realism | **(process, activity, sensor) cell** — median over that cell's curves | those **cells** | **lower** (0 = perfect) |
| **C** | Complete energy profile | **(process, case, sensor)** — one curve against its own real counterpart; no stage-1 collapse, the case *is* the unit | those **units** | **lower** (0 = perfect) |

So the P timing columns and C are per case; the P discovery columns are the only
ones that cannot be, and I aggregates curves into cells first.

Because the timing columns pool cases, processes are **not** equally weighted —
one with more cases counts more. Section 3.P prints the share each carries.

Median at both stages: these errors are bounded below with a long right tail, so
a few bad cases or cells would drag any mean, and a mean of medians is neither
statistic.

**Nothing is reported per process.** The process is an identifier, not a result,
so it never appears as a row or column anywhere below.

Sections: **1** headline tables · **2** LaTeX for the paper · **3** evaluation
counts · **4** full breakdown tables (anonymised).

Names are prefixed `p_` / `i_` / `c_` (or `P_` / `I_` / `C_` for config) so the
three evaluations never collide.

## 0 · Setup

In [ ]:
# ── Shared config ────────────────────────────────────────────────────────────
import warnings, re
import numpy as np, pandas as pd
from pathlib import Path
from IPython.display import display, Markdown, HTML
warnings.filterwarnings('ignore')

RESULTS_ROOT = Path('..') / 'results'

# Identity labels (process / sensor / activity / case) become shuffled numbers.
# One seed for all three evaluations, so a label means the same thing everywhere
# the underlying label sets agree.
ANONYMISE = True
ANON_SEED = 20260730

SAVE_LATEX = True

# Experiments. P reads a different run from I/C on purpose — keep them separate.
EXPERIMENT = 1

P_SPLIT = 'test'      # split reported by the process tables
P_AGG   = 'median'    # aggregation across processes
I_SPLIT = 'TEST'      # 'TEST' | 'TRAIN'

# Model selection ALWAYS happens on TRAIN — selecting on the reported split would
# let a model be chosen for fitting the evaluation data.
SELECTION_SPLIT            = 'train'
SELECTION_METRIC_BASES     = ['conformance_metrics_fitness',
                              'conformance_metrics_precision',
                              'conformance_metrics_generalization',
                              'conformance_metrics_simplicity']
SELECTION_HIGHER_IS_BETTER = True     # quality scores: best = argmax


def _anon_map(values, prefix, seed):
    """Sorted real labels -> '<prefix> <n>', n a fixed random permutation of 1..N."""
    labels = sorted(pd.Series(values).dropna().astype(str).unique())
    order  = np.random.default_rng(seed).permutation(len(labels)) + 1
    return {lab: f'{prefix} {n}' for lab, n in zip(labels, order)}


def _tex_esc(s):
    return (str(s).replace('\\', r'\textbackslash ').replace('&', r'\&')
                  .replace('_', r'\_').replace('%', r'\%').replace('#', r'\#'))


def _latest_run(experiment):
    """Newest run dir for this experiment, by its TIMESTAMP.

    The name must be exactly 'experiment_<n>_<YYYYMMDD>_<HHMMSS>'. A plain
    startswith + sorted() picked up the retired 'experiment_1_old_...' and
    'experiment_1_failed_...' dirs and, because 'o'/'f' sort after the digit
    that starts a timestamp, returned the OLD run as the newest one.
    """
    pat  = re.compile(rf'^experiment_{experiment}_(\d{{8}}_\d{{6}})$')
    runs = sorted((m.group(1), d) for d in RESULTS_ROOT.iterdir()
                  if d.is_dir() and (m := pat.match(d.name)))
    assert runs, f'No runs for experiment {experiment}'
    return runs[-1][1]


def _parse_mode(m):
    """'petri_net_<model>[_ml_plus_*]' -> (model, time-prediction variant)."""
    r = str(m)
    if not r.startswith('petri_net_'):
        return r, 'baseline'
    r = r[len('petri_net_'):]
    if r.endswith('_ml_plus_global'):  return r[:-len('_ml_plus_global')],  'ml_global'
    if r.endswith('_ml_plus_per_act'): return r[:-len('_ml_plus_per_act')], 'ml_local'
    return r, 'baseline'


def _select_best_miner(df):
    """Combined-best per process: the pipeline's recorded TRAIN verdict, else the
    identical rule re-derived here. 'combined' is that verdict, not a miner, so it
    never competes against the miners it was selected from."""
    pre  = SELECTION_SPLIT.lower() + '_'
    cols = [pre + b for b in SELECTION_METRIC_BASES if (pre + b) in df.columns]
    assert cols, f'no {pre}* selection columns found'
    cands = sorted(set(df['model'].unique()) - {'alpha', 'budget', 'combined'})
    choice = {}
    if 'selected_mining_algorithm' in df.columns:
        cmb = df[(df['model'] == 'combined') & df['selected_mining_algorithm'].notna()]
        choice = cmb.groupby('process')['selected_mining_algorithm'].first().to_dict()
    best = {}
    for proc, g in df.groupby('process'):
        if proc in choice:
            best[proc] = choice[proc]; continue
        gc = g[g['model'].isin(cands)]
        if gc.empty:
            best[proc] = None; continue
        score = gc.groupby('model')[cols].mean().mean(axis=1)
        best[proc] = score.idxmax() if SELECTION_HIGHER_IS_BETTER else score.idxmin()
    src = 'pipeline, on TRAIN' if choice else f'notebook fallback, on {SELECTION_SPLIT.upper()}'
    print(f'Combined-best by {src} | candidates: {cands}')
    for p, m in best.items():
        print(f'  {p}: {m}')
    return best, cands

### 0.P · Process discovery — load

In [ ]:
# ── P: load process_eval_results, define the metric columns ──────────────────
p_run = _latest_run(EXPERIMENT)
print('P run:', p_run.name)
pdf = pd.read_parquet(p_run / 'process_eval_results.parquet')
_p_sp = P_SPLIT.lower() + '_'
assert any(c.startswith(_p_sp) for c in pdf.columns), f'No {_p_sp}* columns found'
pdf['model'], pdf['time_pred'] = zip(*pdf['mode'].map(_parse_mode))
pdf = pdf.copy()

# Discovery quality FIRST (higher = better), then timing errors (lower = better).
# Every metric carries its direction so highlighting and LaTeX bolding pick the
# right end. Columns: (parquet base, label, LaTeX header, decimals, direction).
_p_dur  = ('duration_metrics_activity_duration_wape'
           if (_p_sp + 'duration_metrics_activity_duration_wape') in pdf.columns
           else 'duration_metrics_activity_duration_error')
_p_span = ('duration_metrics_case_span_wape'
           if (_p_sp + 'duration_metrics_case_span_wape') in pdf.columns
           else 'duration_metrics_case_span_error')

P_SHOW_EVENT_RATIO = True
P_METRICS = [
    ('conformance_metrics_fitness',            'Fitness',        r'Fitness',                      3, 'max'),
    ('conformance_metrics_precision',          'Precision',      r'Precision',                    3, 'max'),
    ('conformance_metrics_generalization',     'Generalization', r'\makecell{General-\\ization}', 3, 'max'),
    ('conformance_metrics_simplicity',         'Simplicity',     r'Simplicity',                   3, 'max'),
]
if P_SHOW_EVENT_RATIO:
    P_METRICS.append(
        ('basic_metrics_event_count_error',    'Evt-Ratio Err',  r'\makecell{Evt-Ratio\\Err}',    3, 'min'))
P_METRICS += [
    (_p_dur,                                   'Activity duration WAPE (%)',   r'\makecell{Activity\\duration\\WAPE (\%)}',    1, 'min'),
    ('duration_metrics_activity_duration_mae', 'Activity duration MAE (min)',  r'\makecell{Dur MAE\\(min)}',    1, 'min'),
    (_p_span,                                  'Lead time WAPE (%)',   r'\makecell{Lead time\\WAPE (\%)}',    1, 'min'),
    ('duration_metrics_case_span_mae',         'Lead time MAE (min)', r'\makecell{Lead time\\MAE (min)}',   1, 'min'),
]
# Dropped: 'Activity duration MAE (min)' is an unweighted mean of absolute minute errors over
# the activity types a case has in COMMON between real and simulated, and each
# method is scored on its OWN intersection. A looser net fires more of the short
# activities, and a short activity can only contribute a small absolute error, so
# the mean is dragged down by composition rather than accuracy -- Alpha "wins" the
# column while losing every scale-free one. Restricting both to the same
# activities reverses it (Alpha 2.04 vs best-net 1.64 min). 'Activity duration WAPE (%)'
# measures the same quantity normalised by the real durations, so it is immune and
# is what the table reports instead.
# 'Lead time MAE (min)' goes for the same reason: case spans run 134 -> 1364 min across
# processes, so pooling absolute minutes mixes incomparable scales (Alpha wins it in
# 4 of 6 processes yet loses pooled). Its standardised form is 'Lead time WAPE (%)', which
# the table keeps. Every error column that survives is now a normalised error,
# median over equally-weighted cases.
P_EXCLUDE_METRICS = ['Activity duration MAE (min)', 'Lead time MAE (min)']

P_METRICS   = [m for m in P_METRICS
               if (_p_sp + m[0]) in pdf.columns and m[1] not in P_EXCLUDE_METRICS]
P_COL_OF    = {l: _p_sp + b for (b, l, t, d, dr) in P_METRICS}
P_LABELS    = [l  for (b, l, t, d, dr) in P_METRICS]
P_TEX_HDR   = {l: t  for (b, l, t, d, dr) in P_METRICS}
P_DECIMALS  = {l: d  for (b, l, t, d, dr) in P_METRICS}
P_DIRECTION = {l: dr for (b, l, t, d, dr) in P_METRICS}


def p_best_of(series, lbl):
    """Direction-aware best of a column slice; None when nothing is finite."""
    v = series.dropna()
    if v.empty:
        return None
    return v.max() if P_DIRECTION[lbl] == 'max' else v.min()


def p_fmt(v, lbl):
    return '' if pd.isna(v) else f'{v:.{P_DECIMALS[lbl]}f}'


print(f'{len(pdf)} rows | processes: {sorted(pdf["process"].unique())}')
print('Discovery:', [l for l in P_LABELS if P_DIRECTION[l] == 'max'])
print('Timing   :', [l for l in P_LABELS if P_DIRECTION[l] == 'min'])

In [ ]:
# ── P: resolve the methods, then assemble BOTH aggregation levels ────────────
p_best_miner, _p_cands = _select_best_miner(pdf)


def p_model_for(proc, pm):
    if pm == 'Alpha':         return 'alpha'
    if pm == 'Budget':        return 'budget'
    if pm == 'Combined-best': return p_best_miner.get(proc)
    return None


P_PM_ORDER   = ['Alpha', 'Combined-best', 'Budget']
P_TIME_ORDER = ['baseline', 'ml_global', 'ml_local']
_P_TIME_SUFFIX = {'baseline': '', 'ml_global': '_ml_plus_global',
                  'ml_local': '_ml_plus_per_act'}

# ── Level 1: DISCOVERY quality — one value per process ───────────────────────
# Fitness / Precision / Generalization / Simplicity are MODEL-level: the real log
# replayed against the discovered net, per station, averaged. They have no
# per-case meaning, so they can only be aggregated across processes.
_recs = []
for proc, g in pdf.groupby('process'):
    for pm in P_PM_ORDER:
        mdl = p_model_for(proc, pm)
        if mdl is None:
            continue
        for _, row in g[g['model'] == mdl].iterrows():
            if row['time_pred'] not in P_TIME_ORDER:
                continue
            rec = {'process': proc, 'Process method': pm, 'Time approach': row['time_pred']}
            rec.update({l: row[c] for l, c in P_COL_OF.items()})
            _recs.append(rec)
p_long = pd.DataFrame(_recs)

# ── Level 2: TIMING — one row per CASE ───────────────────────────────────────
# process_eval_per_case.parquet holds the raw per-case sample the pipeline's
# per-process medians were computed from: one row per (process, mode, split,
# case_id). Pooling it directly makes every CASE one unit of the median, instead
# of every process. The two are not the same: case counts are very uneven, so a
# process with many cases now carries proportionally more weight.
P_TIMING_UNIT = 'case'        # 'case' pools per_case rows; 'process' medians the
                              # pipeline's per-process values as before
p_cases = pd.read_parquet(p_run / 'process_eval_per_case.parquet')
p_cases = p_cases[p_cases['split'].astype(str).str.upper() == P_SPLIT.upper()]

_rows = []
for pm in P_PM_ORDER:
    for tp in P_TIME_ORDER:
        for proc in sorted(pdf['process'].unique()):
            mdl = p_model_for(proc, pm)
            if mdl is None:
                continue
            mode = f'petri_net_{mdl}{_P_TIME_SUFFIX[tp]}'
            g = p_cases[(p_cases['process'] == proc) & (p_cases['mode'] == mode)]
            if g.empty:
                print(f'  WARNING: no per-case rows for {proc} / {pm} / {tp} ({mode})')
                continue
            g = g.copy()
            g['Process method'], g['Time approach'] = pm, tp
            _rows.append(g)
p_case_long = pd.concat(_rows, ignore_index=True)

# Which per-case column each reported timing metric comes from. EVERY column is
# the MEDIAN over cases, so every case counts exactly once.
#
# The span column is a pooled WAPE over all test cases: sum(case_span_mae) /
# sum(span_real) * 100. Note it therefore weights each case by its real lead
# time (process_4_1 + 4_2 carry most of the minutes); the column is named
# 'Lead time WAPE (%)'. 'median_pct' = median of a ratio, as a percentage;
# 'pooled_wape' = ratio of sums over the group.
P_PER_CASE = {
    'basic_metrics_event_count_error':          ('evt_ratio_err',  'median'),
    'duration_metrics_activity_duration_wape':  ('dur_wape',       'median'),
    'duration_metrics_activity_duration_error': ('dur_err_activ',  'median'),
    'duration_metrics_activity_duration_mae':   ('dur_mae',        'median'),
    'duration_metrics_case_span_wape':          ('case_span_mae',  'pooled_wape'),
    'duration_metrics_case_span_error':         ('case_span_mae',  'pooled_wape'),
    'duration_metrics_case_span_mae':           ('case_span_mae',  'median'),
}
P_DISC_LABELS = [l for l in P_LABELS if P_DIRECTION[l] == 'max']
P_TIME_LABELS = [l for l in P_LABELS if P_DIRECTION[l] == 'min']
_p_base_of = {l: b for (b, l, t, d, dr) in P_METRICS}
_p_unmapped = [l for l in P_TIME_LABELS if _p_base_of[l] not in P_PER_CASE]
assert not _p_unmapped, f'no per-case column mapped for {_p_unmapped}'

print(f'p_long (discovery): {len(p_long)} rows | {p_long["process"].nunique()} processes')
# case_id repeats across processes, so the timing unit is the (process, case)
# PAIR — counting case_id alone would undercount it.
P_N_CASES = len(p_case_long[['process', 'case_id']].drop_duplicates())
print(f'p_case_long (timing): {len(p_case_long):,} rows | '
      f'{P_N_CASES:,} distinct (process, case) units')
print('cases per (method, time approach):')
print(p_case_long.groupby(['Process method', 'Time approach'])['case_id'].size().to_string())

### 0.I · Individual profile realism — load

In [ ]:
# ── I: config ────────────────────────────────────────────────────────────────
I_EXCLUDE_APPROACHES = ['Exemplar (real curve)', 'Exemplar (no DTW)',
                        'Exemplar + DTW (warped)']   # raw names, dropped everywhere
I_MIN_CURVE_POINTS   = 5      # curve-length floor
I_LEVEL_CELL_COVERAGE = True  # restrict to the cells every approach is scored on

# err = |f(pred) - f(real)| / mean|f(real)| per (process, sensor): target 0 everywhere.
I_REALISM_SPECS = {
    'Sum':       ('sum_pred',   'sum_real',   0.0),   # total energy over the curve
    'Max':       ('max_pred',   'max_real',   0.0),   # peak — sizing and tariffs
    'Mean':      ('mean_pred',  'mean_real',  0.0),   # average load level = the bias
    'Std':       ('std_pred',   'std_real',   0.0),   # spread — the flatness measure
    'Roughness': ('rough_pred', 'rough_real', 0.0),   # jaggedness — catches amplitude
                                                      # reached by noise, not dynamics
}
I_REALISM_TARGET = {k: v[2] for k, v in I_REALISM_SPECS.items()}

# Paper naming: the proposed method is 'Step DTW'; every other row is named by what
# it IS, not by what was removed -- the ablation logic belongs in the caption.
I_METHOD_RENAME = {
    'Step DTW smooth + ML + Ext.':    'ML Step DTW (proposed)',
    'Step DTW + ML + Ext.':           'Step DTW (step gains)',
    'ML + Ext. Factors':              'ML DTW (no steps)',
    'DTW + ML + Ext. Factors':        'ML DTW (no steps)',
    'ML DTW':                         'Step DTW w/o segments and ext.',
    'DTW + ML':                       'Step DTW w/o segments and ext.',
    'ML only (no DTW)':               'ML (no DTW)',
    'Median per Activity & Sensor':   'Median per Activity & Sensor',
    'Median per activity and sensor': 'Median per Activity & Sensor',
    'Baseline':                       'Sensor median',
    'DTW + Seq2Seq + Ext. Factors':   'Seq2Seq DTW (aligned)',
    'Seq2Seq IOM (DTW-selected)':     'Seq2Seq (DTW-scored)',
    'DTW + Seq2Seq':                  'Seq2Seq (aligned, w/o ext.)',
    'Seq2Seq only (no DTW)':          'Seq2Seq (unaligned)',
    'Exemplar (real curve)':          'Exemplar (real curve)',
}

i_run = _latest_run(EXPERIMENT)
print('I run:', i_run.name)
i_raw = pd.read_parquet(i_run / 'curve_eval_results.parquet')
i_raw = i_raw[i_raw['Split'] == I_SPLIT]
if I_EXCLUDE_APPROACHES:                       # exclusion uses the RAW names
    i_raw = i_raw[~i_raw['Approach'].isin(I_EXCLUDE_APPROACHES)]

_i_real_path = i_run / 'real_test_curves.parquet'
if 'y_pred' in i_raw.columns and 'y_true' not in i_raw.columns and _i_real_path.exists():
    _i_real = pd.read_parquet(_i_real_path)
    _i_key  = [c for c in ['Process', 'Sensor', 'Activity', 'Instance', 'Split']
               if c in i_raw.columns and c in _i_real.columns]
    i_raw = (i_raw.merge(_i_real[_i_key + ['y_real']], on=_i_key, how='left')
                  .rename(columns={'y_real': 'y_true'}))
    print(f'Joined {len(_i_real):,} real curves on {" + ".join(_i_key)} '
          f'({i_raw["y_true"].notna().mean()*100:.1f}% matched)')

# Anonymise BEFORE any grouping. Sensors are keyed on (process, sensor), so a name
# shared by two processes still gets two distinct labels.
I_ANON_MAPS = {}
if ANONYMISE:
    _i_pair = i_raw['Process'].astype(str) + ' | ' + i_raw['Sensor'].astype(str)
    for _i, _col in enumerate(['Process', 'Activity']):
        if _col in i_raw.columns:
            I_ANON_MAPS[_col] = _anon_map(i_raw[_col], _col, ANON_SEED + _i)
            i_raw[_col] = i_raw[_col].astype(str).map(I_ANON_MAPS[_col])
    I_ANON_MAPS['Sensor'] = _anon_map(_i_pair, 'Sensor', ANON_SEED + 2)
    i_raw['Sensor'] = _i_pair.map(I_ANON_MAPS['Sensor'])
    print('Anonymised: ' + ', '.join(f'{c} ({len(m)})' for c, m in I_ANON_MAPS.items()))

In [ ]:
# ── I: realism errors, population levelling, cell-level aggregation ──────────
I_CELL_KEY = ['Process', 'Activity', 'Sensor']
_i_before  = i_raw.groupby('Approach').size()

for _side in ('real', 'pred'):
    if f'mean_{_side}' in i_raw.columns and 'N' in i_raw.columns:
        i_raw[f'sum_{_side}'] = (pd.to_numeric(i_raw[f'mean_{_side}'], errors='coerce')
                                 * pd.to_numeric(i_raw['N'], errors='coerce'))

_i_specs = {k: v for k, v in I_REALISM_SPECS.items()
            if v[0] in i_raw.columns and v[1] in i_raw.columns}
_i_missing = sorted({c for s in I_REALISM_SPECS.values() for c in s[:2]} - set(i_raw.columns))

if _i_specs:
    for _name, (_num, _den, _tgt) in _i_specs.items():
        p = pd.to_numeric(i_raw[_num], errors='coerce')
        r = pd.to_numeric(i_raw[_den], errors='coerce')
        # Scale = mean |real| of that (process, sensor), over DISTINCT curves.
        _uniq  = i_raw.drop_duplicates(['Process', 'Sensor', 'Instance'])
        _by_cell = (pd.to_numeric(_uniq[_den], errors='coerce').abs()
                      .groupby([_uniq['Process'], _uniq['Sensor']]).mean())
        _scale = pd.Series(pd.MultiIndex.from_arrays([i_raw['Process'], i_raw['Sensor']])
                             .map(_by_cell).to_numpy(dtype=float), index=i_raw.index)
        i_raw[_name] = np.where(np.isfinite(_scale) & (_scale > 1e-6),
                                (p - r).abs() / _scale, np.nan)
    I_ACTIVE = [m for m in I_REALISM_SPECS if m in _i_specs and i_raw[m].notna().any()]
    print('Realism metrics:', I_ACTIVE)
    if _i_missing:
        print(f'  (absent from this run, skipped: {_i_missing})')
else:
    I_ACTIVE = []
    print('No realism metrics in this run — re-run the pipeline to write the '
          'per-curve shape statistics into curve_eval_results.parquet.')

# Levelling 1 — curve length. Levelling 2 — cell coverage, applied AFTER the
# length floor, because dropping short curves can itself empty a cell for one
# approach and not another.
if I_MIN_CURVE_POINTS:
    i_raw = i_raw[i_raw['N'] >= I_MIN_CURVE_POINTS]
if I_LEVEL_CELL_COVERAGE:
    _cells_of = {a: set(map(tuple, g[I_CELL_KEY].drop_duplicates().to_numpy()))
                 for a, g in i_raw.groupby('Approach')}
    _shared = set.intersection(*_cells_of.values()) if _cells_of else set()
    i_raw = i_raw[i_raw[I_CELL_KEY].apply(tuple, axis=1).isin(_shared)]
    print(f'Cell coverage levelled to the {len(_shared)} (process, activity, sensor) '
          f'cells every approach is scored on')
    for _a, _n in sorted({a: len(c - _shared) for a, c in _cells_of.items() if c - _shared}.items()):
        print(f'  {_a}: dropped {_n} cells no other approach has')

i_levelling = pd.DataFrame({'curves_raw': _i_before,
                            'curves_used': i_raw.groupby('Approach').size()}).fillna(0).astype(int)
i_levelling['dropped'] = i_levelling['curves_raw'] - i_levelling['curves_used']
if i_levelling['curves_used'].nunique() > 1:
    print('curve counts still differ after levelling — not like-for-like.')

i_raw['Approach'] = i_raw['Approach'].replace(I_METHOD_RENAME)   # display names from here

# Stage 1: one value per (process, activity, sensor, approach) = median over its
# curves. Median at BOTH stages — these errors are bounded below with a long right
# tail, so a few bad cells would drag any mean, and a mean of medians is neither.
i_combo = (i_raw.groupby(['Process', 'Activity', 'Sensor', 'Approach'])[I_ACTIVE]
                .median().reset_index())
print(f'{len(i_raw):,} curves -> {len(i_combo):,} (process, activity, sensor, approach) cells | '
      f'{i_raw["Approach"].nunique()} approaches')

In [ ]:
# ── I: scorecard + table helpers ─────────────────────────────────────────────
def i_realism_deviation(values, metric):
    """Distance to target. Every realism metric is a normalised ABSOLUTE error with
    target 0, so this is the value itself — kept as a function so a signed metric
    added later cannot silently make the ranking bold the worst cell."""
    v = pd.to_numeric(pd.Series(values), errors='coerce')
    return (v - I_REALISM_TARGET.get(metric, 0.0)).abs()


def i_realism_scorecard(df_combo, how='median', metrics=None):
    """Stage 2: collapse the cells into one row per approach, plus 'Overall' = the
    plain average across the metric columns. Rows best-first."""
    metrics = list(metrics if metrics is not None else I_ACTIVE)
    t = df_combo.groupby('Approach')[metrics].agg(how)
    if len(t.columns):
        t['Overall'] = t.mean(axis=1)
        t = t.sort_values('Overall')
    return t


def i_style_realism(t, how='median'):
    def _best(col):
        d = i_realism_deviation(col, col.name)
        return ['font-weight:700;background-color:#d6ecff;'
                if (pd.notna(v) and pd.notna(d.min()) and abs(v - d.min()) < 1e-12)
                else '' for v in d]
    return (t.style.format({m: '{:.3f}' for m in t.columns}, na_rep='—')
             .apply(_best, axis=0)
             .set_caption(f'{I_SPLIT} — {how} over process x activity x sensor; '
                          f'target 0, lower = better'))

### 0.C · Complete energy profile — load

In [ ]:
# ── C: config ────────────────────────────────────────────────────────────────
# Rows of every C table (order = row order).
C_METHODS = {
    'Baseline':          True,   # per-SENSOR median curve (naive curve generator)
    'Alpha':             True,   # alpha-miner net     + C_CURVE_APPROACH
    'Combined-best':     True,   # best discovered net + C_CURVE_APPROACH
    'Budget':            True,   # best net + duration budgeting + C_CURVE_APPROACH
    'Schedule-direct':   True,   # real schedule, curves taken directly
    'Schedule-step':     True,   # real schedule, Step DTW curves — the ablation
                                 # that isolates the process model from the curve
                                 # method (Schedule-direct changes both at once)
    'Profile-generator': True,   # stochastic profile generator
}
# Columns of every C table.
C_FEATURE_TOGGLE = {'total': True, 'peak': True, 'mean': True,
                    'std': True, 'roughness': True}

C_CURVE_APPROACH          = 'ml_step_dtw_smooth'
C_BASELINE_CURVE_APPROACH = 'baseline'
C_BASELINE_PROCESS_TYPE   = 'Budget'   # Baseline uses this net; only its CURVES are
                                       # the naive median, so the difference is
                                       # attributable to the curve generator alone.
C_DURATION_MODE = 'ml_local'           # fixed for every process ('best_by_mae' picks
                                       # per process on SELECTION_SPLIT instead)
C_DURATION_SELECT_METRIC = 'duration_metrics_activity_duration_mae'

C_SCHEDULE_SERIES = {'schedule': 'Schedule-direct', 'schedule_step': 'Schedule-step',
                     'stochastic': 'Profile-generator'}
C_PROCESS_TYPES = {k: C_METHODS.get(k, False) for k in ('Alpha', 'Combined-best', 'Budget')}
C_EXTRA_METHODS = {k: C_METHODS.get(k, False)
                   for k in ('Schedule-direct', 'Schedule-step', 'Profile-generator')}
C_SHOW_BASELINE = C_METHODS.get('Baseline', False)

C_ALL_FEATURES = ['total', 'peak', 'mean', 'std', 'roughness']
C_FEATURES     = [f for f in C_ALL_FEATURES if C_FEATURE_TOGGLE.get(f)]
C_FEAT_LABEL   = {'total': 'Sum', 'peak': 'Max', 'mean': 'Mean',
                  'std': 'Std', 'roughness': 'Roughness'}


# 'total' is the plain sum of the samples: it treats every sample as one minute
# wide, so it carries a small grid-dependent offset (real dt = 1.00 min,
# petri-net ~1.04, schedule / stochastic ~1.21) on top of the prediction error.
def c_curve_features(v):
    v = np.asarray(v, float)
    v = v[np.isfinite(v)]
    if v.size < 4:
        return None
    return {'total':     float(np.nansum(v)),
            'peak':      float(np.nanmax(v)),
            'mean':      float(np.nanmean(v)),
            'std':       float(np.nanstd(v)),
            'roughness': float(np.nanmean(np.abs(np.diff(v))))}


def c_features_long(df_curves, method_label):
    rows = []
    for (sen, cid), g in df_curves.groupby(['sensor', 'case_id']):
        f = c_curve_features(g.sort_values('t_minutes')['value'].to_numpy())
        if f:
            f.update(sensor=sen, case_id=cid, series=method_label); rows.append(f)
    return rows


c_run = _latest_run(EXPERIMENT)
print('C run:', c_run.name)

In [ ]:
# ── C: resolve process type -> concrete simulation mode per process ──────────
c_pe = pd.read_parquet(c_run / 'process_eval_results.parquet')
c_pe['model'], c_pe['time_pred'] = zip(*c_pe['mode'].map(_parse_mode))
c_best_miner, _c_cands = _select_best_miner(c_pe)
assert _c_cands, 'no miner candidates left after excluding alpha/budget/combined'

_C_TIME_SUFFIX = {'baseline': '', 'ml_global': '_ml_plus_global',
                  'ml_local': '_ml_plus_per_act'}
# C_DURATION_MODE='best_by_mae' picks the variant per process by activity-duration
# MAE. A SELECTION, so it reads the TRAIN column. MAE not WAPE: the comparison is
# within one process, so the scale is constant and MAE stays in minutes.
_c_dur_col = SELECTION_SPLIT.lower() + '_' + C_DURATION_SELECT_METRIC
assert _c_dur_col in c_pe.columns, f'{_c_dur_col} missing'


def c_model_for(proc, ptype):
    return {'Alpha': 'alpha', 'Budget': 'budget'}.get(ptype) or c_best_miner.get(proc)


def c_mode_for(proc, ptype):
    model = c_model_for(proc, ptype)
    if model is None:
        return None
    if C_DURATION_MODE in _C_TIME_SUFFIX:
        tp = C_DURATION_MODE
    else:
        sub = c_pe[(c_pe['process'] == proc) & (c_pe['model'] == model)]
        tp = (sub.loc[sub[_c_dur_col].idxmin(), 'time_pred']
              if not sub.empty and sub[_c_dur_col].notna().any() else 'baseline')
    return f'petri_net_{model}{_C_TIME_SUFFIX[tp]}'


def _c_suffix(approach):   # the pipeline writes the 'baseline' approach unsuffixed
    return '' if approach in ('baseline', '', None) else f'_{approach}'


_c_curve_sfx    = _c_suffix(C_CURVE_APPROACH)
_c_baseline_sfx = _c_suffix(C_BASELINE_CURVE_APPROACH)

In [ ]:
# ── C: build the feature table (real + every method) ─────────────────────────
_c_ptypes = [t for t in ['Alpha', 'Combined-best', 'Budget'] if C_PROCESS_TYPES.get(t)]
_c_extra  = [s for s, lbl in C_SCHEDULE_SERIES.items() if C_EXTRA_METHODS.get(lbl)]
C_METHOD_ORDER = ((['Baseline'] if C_SHOW_BASELINE else []) + _c_ptypes
                  + [C_SCHEDULE_SERIES[s] for s in _c_extra])

_rows = []
for proc in sorted(c_pe['process'].unique()):
    for ptype in _c_ptypes:
        mode = c_mode_for(proc, ptype)
        fp = c_run/'complete_curve_eval_results'/proc/(mode or '')/f'predicted_curves{_c_curve_sfx}.parquet'
        # Warned, not skipped quietly: a missing mode empties the method's row,
        # which is indistinguishable from a method that was switched off.
        if not mode or not fp.exists():
            print(f'  WARNING: {ptype} curves missing for {proc} ({mode}) — row will be empty')
            continue
        for r in c_features_long(pd.read_parquet(fp).query("series == 'predicted'"), ptype):
            r['process'] = proc; _rows.append(r)
    if C_SHOW_BASELINE:
        bmode = c_mode_for(proc, C_BASELINE_PROCESS_TYPE)
        bfp = c_run/'complete_curve_eval_results'/proc/(bmode or '')/f'predicted_curves{_c_baseline_sfx}.parquet'
        if bmode and bfp.exists():
            for r in c_features_long(pd.read_parquet(bfp).query("series == 'predicted'"), 'Baseline'):
                r['process'] = proc; _rows.append(r)
        else:
            print(f'  WARNING: Baseline curves missing for {proc} ({bmode}) — row will be empty')
    sfp = c_run / 'schedule_profile_eval_results' / proc / 'predicted_curves.parquet'
    if not sfp.exists():
        continue
    sdf = pd.read_parquet(sfp)
    for r in c_features_long(sdf[sdf['series'] == 'real'], 'real'):
        r['process'] = proc; _rows.append(r)
    for s in _c_extra:
        for r in c_features_long(sdf[sdf['series'] == s], C_SCHEDULE_SERIES[s]):
            r['process'] = proc; _rows.append(r)
c_feat = pd.DataFrame(_rows)

# Restrict to the sensors common to real + every method within each process.
_keep = []
for proc, g in c_feat.groupby('process'):
    sets = [set(g[g.series == m]['sensor'].unique())
            for m in (['real'] + C_METHOD_ORDER) if m in set(g.series)]
    common = set.intersection(*sets) if sets else set()
    _keep.append(g[g['sensor'].isin(common)])
c_feat = pd.concat(_keep, ignore_index=True)
print('c_feat rows:', len(c_feat), '| methods:', C_METHOD_ORDER)
_c_missing = [m for m in C_METHOD_ORDER if m not in set(c_feat['series'])]
if _c_missing:
    print(f'WARNING: NO DATA for: {_c_missing} — absent from every table')

In [ ]:
# ── C: one row per (process, case, sensor, method, feature) ──────────────────
# AGGREGATION UNIT = (process, case, sensor). Each case is compared to its OWN
# real counterpart, so a method cannot match the marginal distribution while being
# wrong on every case. Value = |f(pred) - f(real)| / mean|f(real)|, normalised per
# sensor so magnitudes stay comparable across sensors.
_recs = []
for (proc, sen), g in c_feat.groupby(['process', 'sensor']):
    real_g = g[g.series == 'real'].drop_duplicates('case_id').set_index('case_id')
    for feature in C_FEATURES:
        r = real_g[feature].dropna()
        if len(r) < 2:
            continue
        scale = np.nanmean(np.abs(r.to_numpy())) + 1e-9
        # Degenerate-scale guard: for zero-inflated sensors mean|real| collapses
        # and the relative error explodes. Skip those cells.
        if scale < 1e-6:
            continue
        for m in C_METHOD_ORDER:
            q = (g[g.series == m].drop_duplicates('case_id')
                  .set_index('case_id')[feature].dropna())
            shared = r.index.intersection(q.index)
            if len(shared) == 0:
                continue
            for cid, v in ((q.loc[shared] - r.loc[shared]).abs() / scale).items():
                _recs.append({'process': proc, 'sensor': sen, 'case_id': cid,
                              'method': m, 'feature': feature, 'rel_err': float(v)})
c_units = pd.DataFrame(_recs)

# Anonymised HERE: the file paths above are built from the real process names, and
# every table below is produced after this point. A different seed per column, else
# two columns with the same number of labels get the same permutation.
C_ANON_MAPS = {}
if ANONYMISE:
    for _i, (_col, _pref) in enumerate([('process', 'Process'), ('sensor', 'Sensor'),
                                        ('case_id', 'Case')]):
        C_ANON_MAPS[_col] = _anon_map(c_units[_col], _pref, ANON_SEED + _i)
        c_units[_col] = c_units[_col].astype(str).map(C_ANON_MAPS[_col])
    print('Anonymised: ' + ', '.join(f'{c} ({len(m)})' for c, m in C_ANON_MAPS.items()))

print(f'c_units: {len(c_units):,} rows | '
      f'{c_units[["process","case_id","sensor"]].drop_duplicates().shape[0]:,} '
      f'distinct (process, case, sensor)')

In [ ]:
# ── C: scorecard + row descriptions ──────────────────────────────────────────
def c_scorecard(df_units):
    t = (df_units.groupby(['method', 'feature'])['rel_err'].median().unstack('feature')
                 .reindex(index=C_METHOD_ORDER, columns=C_FEATURES))
    t.columns = [C_FEAT_LABEL[c] for c in t.columns]
    t['Overall'] = t.mean(axis=1)     # simple average across the metric columns
    t = t.sort_values('Overall')      # best first
    t.index.name = 'Method'
    return t


# The duration predictor is part of the process model: C_DURATION_MODE picks it per
# process, so it is summarised as the majority choice with (n/processes) appended
# when the processes did not all pick the same one.
_C_TIME_LABEL = {'baseline': 'baseline duration', 'ml_global': 'ML_global',
                 'ml_local': 'ml_local'}


def _c_time_variant(ptype):
    picks = []
    for proc in sorted(c_pe['process'].unique()):
        mode = c_mode_for(proc, ptype)
        if not mode:
            continue
        picks.append('ml_global' if mode.endswith('_ml_plus_global') else
                     'ml_local'  if mode.endswith('_ml_plus_per_act') else 'baseline')
    if not picks:
        return ''
    cnt = pd.Series(picks).value_counts()
    return _C_TIME_LABEL[cnt.index[0]] + ('' if len(cnt) == 1 else f' ({cnt.iloc[0]}/{len(picks)})')


# One label per row: the method AND the process model it is generated from.
C_METHOD_LABEL = {
    'Baseline':          'Sensor median (no process model)',
    'Alpha':             f'Alpha Petri net + {_c_time_variant("Alpha")}',
    'Combined-best':     f'Best Petri net + {_c_time_variant("Combined-best")}',
    'Budget':            f'Best Petri net + Budget + {_c_time_variant("Budget")}',
    'Schedule-direct':   'Schedule-direct (no process model)',
    'Schedule-step':     'Schedule-direct, Step DTW (no process model)',
    'Profile-generator': 'Profile-generator (no process model)',
}
C_METHOD_CURVE_GEN = {
    'Baseline':          'Sensor median',
    'Alpha':             'Step DTW',
    'Combined-best':     'Step DTW',
    'Budget':            'Step DTW',
    'Schedule-direct':   'Schedule-only regressor',
    'Schedule-step':     'Step DTW',
    'Profile-generator': 'Stochastic generator',
}


def c_with_description(tbl):
    t = tbl.copy()
    t.insert(0, 'Curve generation', [C_METHOD_CURVE_GEN.get(m, '') for m in t.index])
    t.index = [C_METHOD_LABEL.get(m, m) for m in t.index]
    t.index.name = 'Method (process model)'
    return t


def c_style_score(tbl):
    t = c_with_description(tbl)
    num = [c for c in t.columns if c != 'Curve generation']
    return (t.style.format('{:.3f}', subset=num, na_rep='—')
              .highlight_min(axis=0, subset=num,
                             props='font-weight:700;background-color:#d6ecff;')
              .set_caption('Median per-case error to real over (process, case, sensor) '
                           '— lower = closer to real'))

---
# 1 · Results

The three headline tables. Everything they are made of is in section 4.

Realism / profile columns are all `err = |f(pred) − f(real)| / mean|f(real)|`,
median over units — **0 = perfect, lower = better**; `Overall` is their average.
`Sum` total energy · `Max` peak load · `Mean` bias · `Std` amplitude of the
dynamics · `Roughness` `mean|Δv|`, jaggedness (the two together separate real
dynamics from added noise).

## 1.P · Process discovery + timing

In [ ]:
# ── P: the headline table — two aggregation levels side by side ─────────────
# Discovery columns: median across PROCESSES (model-level, no per-case meaning).
# Timing columns:    pooled across CASES (see P_TIMING_UNIT).
def p_timing_pooled(g):
    # One pooled timing value per metric, from the per-case rows of `g`.
    out = {}
    for lbl in P_TIME_LABELS:
        col, how = P_PER_CASE[_p_base_of[lbl]]
        if how == 'pooled_wape':
            out[lbl] = float(g['case_span_mae'].sum() / g['span_real'].sum() * 100)
            continue
        v = pd.to_numeric(g[col], errors='coerce').replace([np.inf, -np.inf], np.nan)
        out[lbl] = float(v.median()) * (100 if how == 'median_pct' else 1)
    return pd.Series(out)


def p_build_combined(agg='median'):
    idx = pd.MultiIndex.from_tuples(
        [(pm, tp) for pm in P_PM_ORDER for tp in P_TIME_ORDER],
        names=['Process method', 'Time approach'])
    disc = (p_long.groupby(['Process method', 'Time approach'])[P_DISC_LABELS].agg(agg)
                  .reindex(idx))
    if P_TIMING_UNIT == 'case':
        time = (p_case_long.groupby(['Process method', 'Time approach'])
                           .apply(p_timing_pooled).reindex(idx))
    else:
        time = (p_long.groupby(['Process method', 'Time approach'])[P_TIME_LABELS].agg(agg)
                      .reindex(idx))
    return pd.concat([disc, time], axis=1)[P_LABELS].dropna(how='all')


p_combined = p_build_combined(agg=P_AGG)


def p_style(tbl, best, caption):
    def _styler(_):
        out = pd.DataFrame('', index=tbl.index, columns=tbl.columns)
        for idx in tbl.index:
            for col in tbl.columns:
                v, b = tbl.loc[idx, col], best.get(col)
                if b is not None and pd.notna(v) and abs(v - b) < 1e-9:
                    out.loc[idx, col] = 'font-weight:700;background-color:#d6ecff;'
        return out
    fmt = {l: (lambda v, ll=l: p_fmt(v, ll)) for l in tbl.columns}
    return tbl.style.format(fmt).apply(_styler, axis=None).set_caption(caption)


_p_unit = (f'pooled over the {P_N_CASES:,} test cases'
           if P_TIMING_UNIT == 'case' else f'{P_AGG} across processes')
display(p_style(p_combined, {c: p_best_of(p_combined[c], c) for c in p_combined.columns},
                f'Discovery quality (higher = better), {P_AGG} across processes | '
                f'timing errors (lower = better), {_p_unit} — {P_SPLIT} set'))

## 1.I · Individual profile realism

In [ ]:
if not I_ACTIVE:
    display(Markdown('> **Skipped — no realism metrics in this run.**'))
else:
    i_realism_median = i_realism_scorecard(i_combo, how='median')
    display(i_style_realism(i_realism_median))

## 1.C · Complete energy profile

In [ ]:
c_score_all = c_scorecard(c_units)
display(c_style_score(c_score_all))

---
# 2 · LaTeX for the paper

Preamble: `booktabs`, `caption`, `float`, `array`, `tabularx`, `makecell`,
`multirow`. With `SAVE_LATEX` each table is also written under `visuals/`.

## 2.P · Process table

In [ ]:
# ── P: LaTeX layout knobs ────────────────────────────────────────────────────
P_LATEX_TABCOLSEP = '4pt'              # column padding (LaTeX default 6pt)
P_PAPER_STRETCH   = '1.3'
P_PAPER_FONT      = r'\footnotesize'

P_PAPER_ROW = {                        # level-0 row labels of the paper table
    'Alpha':         r'\makecell[c]{Alpha \\ (Baseline)}',
    'Combined-best': r'\makecell[c]{Best \\ Petri Net}',
    'Budget':        r'\makecell[c]{Best \\ Petri Net \\ + Budget}',
}
P_PAPER_HDR = {
    'Fitness':        r'Fitness',
    'Precision':      r'Precision',
    'Generalization': r'\makecell{Generalization}',
    'Simplicity':     r'Simplicity',
    'Evt-Ratio Err':  r'\makecell{Evt-Ratio\\Error}',
    'Activity duration WAPE (%)':   r'\makecell{Activity\\duration\\WAPE (\%)}',
    'Activity duration MAE (min)':  r'\makecell{Activity\\duration\\MAE (min)}',
    'Lead time WAPE (%)':   r'\makecell{Lead time\\WAPE (\%)}',
    'Lead time MAE (min)': r'\makecell{Case \\span MAE\\(min)}',
}
# Decimals used ONLY by the paper table: substring rules first, then exact-label
# overrides, then P_DECIMALS.
P_PAPER_DECIMALS_RULES = [('WAPE', 3)]
P_PAPER_DECIMALS       = {}

P_TEX_IDX_NAME = {'Process method': 'Method',
                  'Time approach': r'\makecell[l]{Time\\approach}'}

def _p_paper_dp(lbl):
    if lbl in P_PAPER_DECIMALS:
        return P_PAPER_DECIMALS[lbl]
    for pat, dp in P_PAPER_DECIMALS_RULES:
        if pat.lower() in str(lbl).lower():
            return dp
    return P_DECIMALS[lbl]


def p_to_paper_table(tbl, caption, label, description):
    """`table*` float in the paper layout: caption on top, booktabs rules, one
    \\multirow block per process method separated by \\cline, description under it.
    Bolding is global per column and direction-aware."""
    n_idx = tbl.index.nlevels
    ncol  = n_idx + len(tbl.columns)
    best  = {c: p_best_of(tbl[c], c) for c in tbl.columns}

    def cell(v, col):
        if pd.isna(v):
            return ''
        s, b = f'{v:.{_p_paper_dp(col)}f}', best[col]
        return (r'\textbf{' + s + '}') if (b is not None and abs(v - b) < 1e-9) else s

    hdr = ([P_TEX_IDX_NAME.get(n, str(n)) for n in tbl.index.names]
           + [P_PAPER_HDR.get(c, P_TEX_HDR[c]) for c in tbl.columns])
    L = [r'\begin{table*}[t]', r'\centering',
         rf'\caption{{{caption}}}', rf'\label{{{label}}}', r'\vspace{-0.5em}',
         rf'\setlength{{\tabcolsep}}{{{P_LATEX_TABCOLSEP}}}',
         rf'\renewcommand{{\arraystretch}}{{{P_PAPER_STRETCH}}}', P_PAPER_FONT,
         r'\begin{tabular}{' + 'l' * n_idx + '|' + '|'.join(['c'] * len(tbl.columns)) + '}',
         r'\toprule', ' & '.join(hdr) + r' \\', r'\midrule']
    for pm in dict.fromkeys(tbl.index.get_level_values(0)):      # keeps P_PM_ORDER
        sub = tbl.xs(pm, level=0, drop_level=False)
        head = rf'\multirow{{{len(sub)}}}{{*}}{{{P_PAPER_ROW.get(pm, _tex_esc(pm))}}}'
        for idx, row in sub.iterrows():
            L.append(' & '.join([head, _tex_esc(idx[-1])]
                                + [cell(row[c], c) for c in tbl.columns]) + r' \\')
            head = ''
        L.append(rf'\cline{{1-{ncol}}}')
    L += [r'\bottomrule', r'\end{tabular}', r'\vspace{0.5em}',
          r'\noindent\raggedright\footnotesize', '', description, '', r'\end{table*}']
    return '\n'.join(L)

In [ ]:
# ── P: the paper table ───────────────────────────────────────────────────────
P_PAPER_DESC = (
    r'Description: Process discovery methods are Alpha (naive miner baseline), '
    r'Best Petri Net (best discovered Petri Net), Best Petri Net + Budget (best Petri '
    r'Net + duration budgeting at simulation time), each crossed with the three '
    r'activity-duration predictors (baseline samples from the fitted statistical '
    r'distributions, ml\_local is a ML model '
    r'per activity and ml\_global is a ML model for all activities). Fitness, '
    r'Precision, Generalization and Simplicity are discovery quality (higher is '
    r'better); Evt-Ratio Error is the relation between total simulated events/number '
    r'of real events, lower is better; and the Activity duration errors are the difference of the '
    r'individual activity durations vs the observed in the test set and the Lead time errors are the '
    r'same but for the total case time (lower is better). \textbf{Bold} = best per column. '
    r'Activity durations are reported as WAPE rather than MAE: MAE averages absolute '
    r'minute errors over the activity types a case shares with its simulation, and '
    r'that set differs per method, so a model that reproduces more short activities '
    r'scores a lower MAE without being more accurate. '
    r'The Lead time WAPE is pooled over all test cases: the summed absolute '
    r'lead-time errors divided by the summed real lead times; '
    r'absolute-minute columns are not reported, since lead times '
    r'differ by an order of magnitude across processes. '
    r'Fitness, Precision, Generalization and Simplicity are model-level and are '
    r'therefore reported as the median across processes; the Activity duration '
    r'WAPE is the median over all test cases, and the Lead time WAPE is pooled over them.')

p_tex_agg = p_to_paper_table(
    p_combined,
    caption=(f'Process modeling and timing accuracy ({P_SPLIT} set results)'),
    label='tab:process_results',
    description=P_PAPER_DESC)
print(p_tex_agg)

if SAVE_LATEX:
    Path('visuals').mkdir(exist_ok=True)
    _f = Path('visuals') / 'process_results.tex'
    _f.write_text('% Process discovery + timing — \\input{} this file.\n' + p_tex_agg + '\n')
    print('\nSaved:', _f)

## 2.I · Individual profile realism table

In [ ]:
# ── I: paper float (table* + minipage + caption + note) ──────────────────────
I_PAPER_HDR      = {'Roughness': 'Roughness'}
I_PAPER_MINIPAGE = '16cm'     # minipage holding caption + tabular
I_PAPER_FIRSTCOL = '6cm'      # p{} width of the method column
I_PAPER_NOTEBOX  = '16cm'     # parbox width of the note under the table
I_PAPER_DECIMALS = 3


def _i_cells(t, dp=3):
    """Formatted strings with the per-column best (closest to target) bolded."""
    s = pd.DataFrame(index=t.index, columns=t.columns, dtype=object)
    for col in t.columns:
        vals = t[col].dropna()
        d    = i_realism_deviation(vals, col).dropna()
        best = vals.get(d.idxmin()) if not d.empty else None
        for idx in t.index:
            v = t.loc[idx, col]
            s.loc[idx, col] = ('' if pd.isna(v) else
                               (r'\textbf{' + f'{v:.{dp}f}' + '}')
                               if (best is not None and abs(v - best) < 1e-6)
                               else f'{v:.{dp}f}')
    return s


def i_to_latex_paper(t, caption, label, note):
    s = _i_cells(t, I_PAPER_DECIMALS)
    body = '\n'.join(' & '.join([_tex_esc(I_METHOD_RENAME.get(idx, idx))]
                                + [s.loc[idx, c] for c in t.columns]) + r' \\'
                     for idx in t.index)
    # Headers are plain text — no \textit / \textbf, matching the P and C tables.
    hdr = ' & '.join(['Method']
                     + [I_PAPER_HDR.get(c, _tex_esc(c)) for c in t.columns])
    return '\n'.join([
        r'\begin{table*}[H]', r'\centering', '',
        rf'\begin{{minipage}}{{{I_PAPER_MINIPAGE}}}', r'\centering', '',
        r'\captionsetup{', r'    justification=centering,',
        r'    singlelinecheck=false,', r'    format=plain', r'}', '',
        rf'\caption{{{caption}}}', rf'\label{{{label}}}', '',
        r'\vspace{-0.5em}', '',
        rf'\begin{{tabular}}{{p{{{I_PAPER_FIRSTCOL}}}|' + '|'.join(['c'] * len(t.columns)) + '}',
        r'\toprule', hdr + r' \\', r'\midrule', body, r'\bottomrule', r'\end{tabular}', '',
        r'\vspace{0.5em}', '',
        rf'\parbox{{{I_PAPER_NOTEBOX}}}{{%', r'\footnotesize', note, r'}', '',
        r'\end{minipage}', '', r'\end{table*}'])


I_REALISM_NOTE = (
    'Curve realism for individual profile prediction, {split} set. Each column is the '
    'median normalised absolute error of that curve property, so 0 is perfect and '
    'lower is better; Overall is their average.\n'
    r'\textbf{{Bold}} marks the best value per column.').format(split=I_SPLIT)

if not I_ACTIVE:
    print('No realism metrics in this run — nothing to emit.')
else:
    i_tex_agg = i_to_latex_paper(
        i_realism_median,
        caption='Curve realism for individual energy-profile prediction.',
        label='tab:individual_profile_realism',
        note=I_REALISM_NOTE)
    print(i_tex_agg)
    if SAVE_LATEX:
        Path('visuals').mkdir(exist_ok=True)
        _f = Path('visuals') / 'individual_profile_realism.tex'
        _f.write_text('% Aggregated realism table — \\input{} this file.\n' + i_tex_agg + '\n')
        print('\nSaved:', _f)

## 2.C · Complete energy profile table

In [ ]:
# ── C: table* + minipage, caption on top, method note at the bottom ──────────
# Typeset with tabularx, so it always fits the text width and the long descriptions
# wrap by themselves — no manual cm widths, no resizebox.
C_LATEX_MINIPAGE   = r'\textwidth'
C_LATEX_NOTE_WIDTH = r'\linewidth'
C_LATEX_FONT       = r'\small'
C_LATEX_TEXT_W     = (1.30, 0.70)   # relative widths of Method / Curve generation; sum = 2
C_GROUP_BY_TYPE    = True           # row order + rules by type; the type itself is not
                                    # printed — the two text columns already say it
C_METHOD_TYPE = {
    'Baseline':          'Baseline',
    'Alpha':             'Process model',
    'Combined-best':     'Process model',
    'Budget':            'Process model',
    'Schedule-direct':   'Schedule-based',
    'Schedule-step':     'Schedule-based',
    'Profile-generator': 'Schedule-based',
}
C_TYPE_ORDER = ['Process model', 'Schedule-based', 'Baseline']

C_METHOD_NOTE = (
    r'\textit{Sensor median (baseline)}: the median curve of each sensor, reused for every case. '
    r'\textit{Alpha Petri Net}: net discovered by the alpha miner. '
    r'\textit{Best Petri Net}: best discovered net per process, selected on the '
    r'training split by the mean of Fitness, Precision, Generalization and Simplicity. '
    r'\textit{Best Petri Net + Budget}: the same net, with each case generated to match '
    r'its predicted total-duration budget. '
    r'\textit{Schedule-direct}: curves placed directly on the real schedule. '
    r'\textit{Schedule-direct, Step DTW}: the same real schedule and the same '
    r'schedule-only inputs, but the \textit{Step DTW} curve predictor instead of the '
    r'schedule-only regressor, so its gap to the Petri-net rows prices the process '
    r'model alone rather than the process model and the curve method together. '
    r'\textit{Profile-generator}: stochastic profile generator. '
    r'The three Petri-net rows use the \textit{Step DTW} curve predictor '
    r'of Evaluation~2. '
    r'Each row names the process model the cases are generated from, including its '
    r'duration predictor (ml\_local = one per activity), and \textit{Curve generation} '
    r'is how the load curve of each activity or case is then produced.')

C_METRIC_NOTE = (r'Cells are the median over (process, case, sensor) of the paired per-case '
                 r'relative error $|f(\mathrm{pred})-f(\mathrm{real})|/\overline{|f(\mathrm{real})|}$. '
                 r'Lower is better; \textbf{bold} = best per column. '
                 r'Overall is the average across the metric columns.')


def _c_row_order(tbl):
    """Rows grouped by type (best Overall first inside each group), or flat."""
    if not C_GROUP_BY_TYPE:
        return list(tbl.index)
    key = 'Overall' if 'Overall' in tbl.columns else tbl.columns[0]
    order = []
    for typ in C_TYPE_ORDER:
        grp = [m for m in tbl.index if C_METHOD_TYPE.get(m) == typ]
        order += sorted(grp, key=lambda m: (pd.isna(tbl.loc[m, key]), tbl.loc[m, key]))
    return order + [m for m in tbl.index if m not in order]


def c_to_latex_score(tbl, caption, label, note_extra=''):
    cols = list(tbl.columns)
    best = {c: tbl[c].dropna().min() for c in cols if tbl[c].notna().any()}

    def cell(m, c):
        v = tbl.loc[m, c]
        if pd.isna(v):
            return '--'
        s = f'{v:.3f}'
        return r'\textbf{' + s + '}' if abs(v - best.get(c, np.inf)) < 1e-9 else s

    order = _c_row_order(tbl)
    _x = [(r'>{\hsize=' + f'{w:g}' + r'\hsize\linewidth=\hsize'
           r'\raggedright\arraybackslash}X') for w in C_LATEX_TEXT_W]
    colfmt = '|'.join(_x) + '|' + '|'.join(['c'] * len(cols))
    # Headers are plain text — no \textbf / \textit, matching the P and I tables.
    head = ('Method (process model) & Curve generation & '
            + ' & '.join(str(c) for c in cols) + r' \\')
    body, i = [], 0
    while i < len(order):
        typ, span = C_METHOD_TYPE.get(order[i], ''), 1
        if C_GROUP_BY_TYPE:
            while i + span < len(order) and C_METHOD_TYPE.get(order[i + span]) == typ:
                span += 1
        for k in range(span):
            mm = order[i + k]
            body.append(' & '.join([_tex_esc(C_METHOD_LABEL.get(mm, mm)),
                                    _tex_esc(C_METHOD_CURVE_GEN.get(mm, ''))]
                                   + [cell(mm, c) for c in cols]) + r' \\')
        if C_GROUP_BY_TYPE and i + span < len(order):
            body.append(r'\midrule')
        i += span
    return '\n'.join([
        r'\begin{table*}[H]', r'\centering', '',
        f'\\begin{{minipage}}{{{C_LATEX_MINIPAGE}}}', r'\centering', '',
        r'\captionsetup{', r'    justification=centering,',
        r'    singlelinecheck=false,', r'    format=plain', r'}', '',
        r'\caption{' + caption + '}', r'\label{' + label + '}', '',
        r'\vspace{-0.5em}', '', C_LATEX_FONT,
        f'\\begin{{tabularx}}{{\\linewidth}}{{{colfmt}}}', r'\toprule', head, r'\midrule',
        *body, r'\bottomrule', r'\end{tabularx}', '', r'\vspace{0.5em}', '',
        f'\\parbox{{{C_LATEX_NOTE_WIDTH}}}{{%', r'\footnotesize',
        C_METHOD_NOTE + (' ' + note_extra if note_extra else ''),
        '}', '', r'\end{minipage}', '', r'\end{table*}'])


c_tex_all = c_to_latex_score(
    c_score_all,
    caption='Complete energy-profile comparison, all processes.',
    label='tab:energy_profile',
    note_extra=C_METRIC_NOTE)
print(c_tex_all)

if SAVE_LATEX:
    Path('visuals').mkdir(exist_ok=True)
    _f = Path('visuals') / 'complete_energy_profile.tex'
    _f.write_text('% Complete energy-profile table — \\input{} this file.\n' + c_tex_all + '\n')
    print('\nSaved:', _f)

---
# 3 · Evaluation counts

How many evaluations each row above is a median over. The comparisons are only
like-for-like if every method is scored on the **same** population; equal counts
are necessary but not sufficient, so the shared set is intersected explicitly.
Any shortfall is flagged.

In [ ]:
def _report_uneven(counts, unit, diag=()):
    """Verdict line for a counts table: every non-diagnostic column must be constant."""
    bad = [c for c in counts.columns if c not in diag and counts[c].nunique(dropna=False) > 1]
    if not bad:
        print(f'OK - like-for-like: every row is scored on the same '
              f'{int(counts.iloc[0, 0])} {unit}')
        return bad
    print(f'WARNING: counts differ — the medians are NOT taken over the same {unit}:')
    for c in bad:
        hi = counts[c].max()
        print(f'   {c}: max {hi}, short rows -> '
              + ', '.join(f'{i}={v}' for i, v in counts[c].items() if v != hi))
    return bad

## 3.P · Process — processes (discovery) and cases (timing)

In [ ]:
# Two populations, because the P table mixes two aggregation levels:
#   processes -- the unit of the four discovery columns
#   cases     -- the unit of every timing column (P_TIMING_UNIT = 'case')
_g = p_long.groupby(['Process method', 'Time approach'])
p_eval_counts = pd.DataFrame({'processes': _g['process'].nunique()})
for _l in P_DISC_LABELS:
    p_eval_counts[_l] = _g[_l].count()
_gc = p_case_long.groupby(['Process method', 'Time approach'])
p_eval_counts['cases'] = _gc['case_id'].size()
for _l in P_TIME_LABELS:
    p_eval_counts[_l] = _gc[P_PER_CASE[_p_base_of[_l]][0]].count()
p_eval_counts = p_eval_counts.reindex(pd.MultiIndex.from_tuples(
    [(pm, tp) for pm in P_PM_ORDER for tp in P_TIME_ORDER
     if (pm, tp) in p_eval_counts.index],
    names=['Process method', 'Time approach'])).fillna(0).astype(int)

display(p_eval_counts)
_report_uneven(p_eval_counts, 'processes / cases')

# Cases per process behind the timing columns. Weighting is NOT uniform: the
# pooled median gives a process with more cases proportionally more influence.
_share = (p_case_long.drop_duplicates(['process', 'case_id'])
                     .groupby('process').size().sort_values(ascending=False))
display(Markdown('**Cases per process** — the weight each process carries in every '
                 'pooled timing number'))
display(pd.DataFrame({'cases': _share,
                      'share of all cases': (_share / _share.sum() * 100).round(1)}))

## 3.I · Individual profile — one (process, activity, sensor) cell

In [ ]:
# curves = per-curve rows of curve_eval_results; cells = the (process, activity,
# sensor) medians they collapse to, and ONE CELL IS ONE UNIT of every I median.
_KEY  = ['Process', 'Activity', 'Sensor']
_sets = {a: set(map(tuple, g[_KEY].drop_duplicates().to_numpy()))
         for a, g in i_combo.groupby('Approach')}
_common = set.intersection(*_sets.values()) if _sets else set()

i_eval_counts = pd.DataFrame({'curves': i_raw.groupby('Approach').size(),
                              'cells':  i_combo.groupby('Approach').size()})
for _m in I_ACTIVE:
    i_eval_counts[f'{_m} cells'] = i_combo.groupby('Approach')[_m].count()
i_eval_counts['shared cells']   = pd.Series({a: len(s & _common) for a, s in _sets.items()})
i_eval_counts['outside shared'] = pd.Series({a: len(s - _common) for a, s in _sets.items()})
i_eval_counts = i_eval_counts.fillna(0).astype(int).sort_index()

display(Markdown('**Population levelled** — curves with $\\geq$ '
                 f'{I_MIN_CURVE_POINTS} samples'
                 + (', on the cells shared by every approach' if I_LEVEL_CELL_COVERAGE else '')))
display(i_levelling)
display(i_eval_counts)
_report_uneven(i_eval_counts, '(process, activity, sensor) cells',
               diag=('shared cells', 'outside shared'))

## 3.C · Complete profile — one (process, case, sensor) unit

In [ ]:
_UKEY = ['process', 'case_id', 'sensor']
_sets = {m: set(map(tuple, g[_UKEY].drop_duplicates().to_numpy()))
         for m, g in c_units.groupby('method')}
_common = set.intersection(*_sets.values()) if _sets else set()

_g = c_units.groupby('method')
c_eval_counts = pd.DataFrame({
    'rows':      _g.size(),
    'units':     _g[_UKEY].apply(lambda d: len(d.drop_duplicates())),
    'processes': _g['process'].nunique(),
    'sensors':   _g.apply(lambda d: len(d[['process', 'sensor']].drop_duplicates())),
    'cases':     _g.apply(lambda d: len(d[['process', 'case_id']].drop_duplicates())),
})
_per_feat = c_units.pivot_table(index='method', columns='feature', values='rel_err',
                                aggfunc='count', fill_value=0)
_per_feat = _per_feat[[f for f in C_FEATURES if f in _per_feat.columns]]
_per_feat.columns = [C_FEAT_LABEL.get(c, c) for c in _per_feat.columns]
c_eval_counts['shared units']   = pd.Series({m: len(s & _common) for m, s in _sets.items()})
c_eval_counts['outside shared'] = pd.Series({m: len(s - _common) for m, s in _sets.items()})
c_eval_counts = (c_eval_counts.join(_per_feat).reindex(C_METHOD_ORDER)
                              .fillna(0).astype(int))
c_eval_counts.index.name = 'Method'

display(c_eval_counts)
_report_uneven(c_eval_counts, '(process, case, sensor) units',
               diag=('shared units', 'outside shared'))

---
# 4 · Full tables (anonymised)

What the section-1 tables of **I** and **C** are made of. Same values and same
statistic — only the grouping is finer, so the coarsest level of each ladder
reproduces the headline table exactly.

**P has no level here.** Its levels would be the process and the case, and this
notebook reports neither as a row — the process is an identifier, not a result.
Section 3.P counts the processes and cases behind every P number instead.

No level is keyed on the process. All remaining identity labels (sensor,
activity, case) are shuffled numbers under `ANONYMISE`: they group the rows
correctly but name nothing. Printed **unrounded** — a zero here is a real zero,
not a rounding artefact.

## 4.I · Individual profile — breakdown ladder

In [ ]:
# Median realism at each level, with the population behind every row. No Process
# column: it is an identifier, not a result, and nothing here is reported per
# process. Nothing merges silently without it — sensor labels are numbered per
# (process, sensor) pair, so a name shared by two processes still gets two rows.
I_LADDER = [('Unit — one (activity, sensor) cell', ['Activity', 'Sensor']),
            ('Activity',                           ['Activity']),
            ('Sensor',                             ['Sensor'])]


def i_breakdown(keys):
    grp = keys + ['Approach']
    t = i_combo.groupby(grp)[I_ACTIVE].median()
    t['Overall'] = t.mean(axis=1)              # same definition as section 1
    t['cells']   = i_combo.groupby(grp).size()  # (process, activity, sensor) units
    t['curves']  = i_raw.groupby(grp).size()    # per-curve evaluations behind them
    return t.sort_values(keys + ['Overall'])


if not I_ACTIVE:
    display(Markdown('> **Skipped — no realism metrics in this run.**'))
else:
    # Bare HTML rather than display(df) or a Styler: at ~2,000 rows both roughly
    # double the size of the .ipynb, and there is no single best row to highlight
    # inside a breakdown anyway.
    for title, keys in I_LADDER:
        t = i_breakdown(keys)                   # full precision, not rounded
        display(Markdown(f'### {title} — {t.index.droplevel(-1).nunique()} groups, '
                         f'{len(t):,} rows ({t["curves"].sum():,} curves)'))
        display(HTML(t.to_html()))

## 4.C · Complete profile — breakdown ladder

In [ ]:
# With ANONYMISE the unit level is keyed on (sensor, case) only: the process column
# is dropped, because even a shuffled 'Process 4' still says which sensors and cases
# belong to the same plant. Without it, the true (process, sensor, case) unit.
C_UNIT_KEY = ['sensor', 'case_id'] if ANONYMISE else ['process', 'sensor', 'case_id']
C_LADDER   = [(f'Unit — one ({", ".join(k.replace("_id", "") for k in C_UNIT_KEY)}) cell',
               C_UNIT_KEY),
              ('Sensor', ['sensor'])]

# Truncates the DISPLAY only (head + tail, as pandas does) — the frame is always
# complete. The unit level is ~16k rows, which is 6 MB of saved HTML shown in full.
C_BREAKDOWN_MAX_ROWS = 400
C_BREAKDOWN_CSV      = False


def c_breakdown(keys):
    grp = keys + ['method']
    t = (c_units.pivot_table(index=grp, columns='feature', values='rel_err',
                             aggfunc='median')
                .reindex(columns=[f for f in C_FEATURES if f in set(c_units['feature'])]))
    t.columns = [C_FEAT_LABEL[c] for c in t.columns]
    t['Overall'] = t.mean(axis=1)                                  # as in the scorecard
    t['units']   = c_units.groupby(grp)[['process', 'case_id', 'sensor']].apply(
                       lambda d: len(d.drop_duplicates()))
    t['values']  = c_units.groupby(grp).size()
    return t.sort_values(keys + ['Overall'])


for title, keys in C_LADDER:
    t = c_breakdown(keys)
    display(Markdown(f'### {title} — {t.index.droplevel(-1).nunique()} groups, '
                     f'{len(t):,} rows'
                     + ('' if C_BREAKDOWN_MAX_ROWS is None or len(t) <= C_BREAKDOWN_MAX_ROWS
                        else f' (showing {C_BREAKDOWN_MAX_ROWS})')))
    display(HTML(t.to_html(max_rows=C_BREAKDOWN_MAX_ROWS)))
    if C_BREAKDOWN_CSV:
        Path('visuals').mkdir(exist_ok=True)
        _f = Path('visuals') / f'complete_profile_breakdown_{"_".join(keys)}.csv'
        t.to_csv(_f); print('Saved:', _f)